# Session 9 — probability and inference: what the data can prove

Data Science & AI. Module 1 Part 2 closes tonight.

Monday you *described* a sample. Tonight's question is harder and pays
better: **what does the sample let us claim about the world?** Two
departments differ by 1.3 years of average service — real pattern, or the
luck of who got sampled? By the end you will answer with the field's
standard tools: the p-value, the confidence interval, the t-test — and
you will have *built* each one from simulation before using the official
functions.

The road: probability (the language) → random variables → the central
limit theorem (the engine) → inference (the payoff) → Bayes (the twist).

## Before you type anything

**Work on your own copy, not on this file**, then **Restart & Run All**.

In [ ]:
# ==========================================================
# Setup. Run this once, then carry on.
# ==========================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

RAW = "https://raw.githubusercontent.com/rejusam/data-science-ai-course/main/"
hr = pd.read_csv(RAW + "data/employee-attrition.csv")

rng = np.random.default_rng(0)
np.set_printoptions(precision=4, suppress=True)
print("rows:", len(hr), "| scipy:", __import__("scipy").__version__)

In [ ]:
from scipy import stats

## 1. Probability: the rules of uncertainty  *(slides 86, 87, 88)*

Probability is a number between 0 and 1 measuring how likely an event
is. The cleanest definition *(slide 88)*: list the **sample space** $\Omega$ —
every outcome an experiment could produce — and then

P(A) = outcomes where A happens ÷ all outcomes.

Sample space of a dice, $\Omega = \{1, 2, 3, 4, 5, 6\} $

Two dice make a sample space of 36 pairs, small enough to enumerate —
Python's sets *(slide 89)* are literally the set operations of the deck:
### Two-Dice Sample Space Grid

| | **1** | **2** | **3** | **4** | **5** | **6** |
|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| **1** | (1,1) | (1,2) | (1,3) | (1,4) | (1,5) | (1,6) |
| **2** | (2,1) | (2,2) | (2,3) | (2,4) | (2,5) | (2,6) |
| **3** | (3,1) | (3,2) | (3,3) | (3,4) | (3,5) | (3,6) |
| **4** | (4,1) | (4,2) | (4,3) | (4,4) | (4,5) | (4,6) |
| **5** | (5,1) | (5,2) | (5,3) | (5,4) | (5,5) | (5,6) |
| **6** | (6,1) | (6,2) | (6,3) | (6,4) | (6,5) | (6,6) |


In [ ]:
omega = {(d1, d2) for d1 in range(1, 7) for d2 in range(1, 7)}
print("size of sample space:", len(omega))

seven  = {p for p in omega if sum(p) == 7}
# omega = {(1, 6), (2, 2), (3, 4), (5, 3)}
# filtered_results = []
# for p in omega:
#     if sum(p) == 7:
#         filtered_results.append(p)
double = {p for p in omega if p[0] == p[1]}

print("P(sum is 7)   :", len(seven), "/ 36 =", round(len(seven) / 36, 3))
print("P(double)     :", len(double), "/ 36 =", round(len(double) / 36, 3))
print("P(7 OR double):", len(seven | double) / 36)     # union
print("P(7 AND double):", len(seven & double) / 36)    # intersection - empty!

In [ ]:
print(seven )
print(double)

No pair is both a seven and a double, so the AND is zero and the OR is
just the sum — the slide-87 rule P(A∪B) = P(A) + P(B) − P(A∩B) with
nothing to subtract.

### What that number means physically

P(sum is 7) = 1/6 is a claim about the *long run*: roll the dice over
and over and the running fraction of sevens wobbles, then settles onto
1/6 — the **law of large numbers**. Watch it happen:

In [ ]:
# n_rolls = 5000

# die1 = rng.integers(1, 7, n_rolls)          # roll the first die, n_rolls times
# die2 = rng.integers(1, 7, n_rolls)          # and the second
# totals = die1 + die2

# was_seven = (totals == 7)                   # True or False, one per roll
# sevens_so_far = np.cumsum(was_seven)        # running count of the Trues
# rolls_so_far = np.arange(1, n_rolls + 1)    # 1, 2, 3, ... n_rolls
# fraction = sevens_so_far / rolls_so_far     # the score after every roll

In [ ]:
n_rolls = 5000

die1 = rng.integers(1, 7, n_rolls)          # 5000 rolls of the first die
die2 = rng.integers(1, 7, n_rolls)          # 5000 rolls of the second
totals = die1 + die2

was_seven = (totals == 7)                   # True or False, one per roll
sevens_so_far = np.cumsum(was_seven)        # running count of the Trues
rolls_so_far = np.arange(1, n_rolls + 1)    # 1, 2, 3, ... 5000
fraction = sevens_so_far / rolls_so_far     # the score after every roll

print("first 8 totals  :", totals[:8])
print("was it a seven? :", was_seven[:8])
print("sevens so far   :", sevens_so_far[:8])
print("fraction so far :", fraction[:8].round(3))

In [ ]:
totals

Put the first eight rolls in a table so you can follow one roll across
the row, and each running total down its column:

In [ ]:
first_8 = pd.DataFrame({
    "die1": die1[:8],
    "die2": die2[:8],
    "total": totals[:8],
    "a seven?": was_seven[:8],
    "sevens so far": sevens_so_far[:8],
    "fraction so far": fraction[:8].round(3),
}, index=rolls_so_far[:8])
first_8.index.name = "roll"

print(first_8)

Find the first `True` and everything explains itself. Up to that roll the
count is stuck on 0, so the fraction is 0. On that roll the count ticks
to 1 and the fraction jumps to 1/6 = 0.167. Then it *falls* — 1/7, then
1/8 — because the sevens stop while the rolls keep coming.

That is all `cumsum` does: keep a running total, counting each `True`
as 1. And the fraction is only ever "sevens so far ÷ rolls so far".

Now plot the last row against the roll number. If you want the first
ten rolls to get as much room as the last thousand, uncomment the
`plt.xscale("log")` line — on the plain axis the wild early part is
squashed against the left edge:

In [ ]:
truth = 6 / 36

plt.figure(figsize=(8, 3.2))
plt.plot(fraction)
plt.axhline(truth, color="red", linestyle="--", label="6/36, the true answer")
plt.xscale("log")
plt.xlabel("rolls so far")
plt.ylabel("fraction that were sevens")
plt.legend()
plt.title("Wild at 10 rolls, settled by 5,000")
plt.show()

print("after {} rolls, the fraction is {:.4f}".format(n_rolls, fraction[-1]))
print("the true answer is           {:.4f}".format(truth))

Read the two halves of that plot as the two faces of probability:

- **early**: anything can happen in ten rolls — probability promises
  nothing about a single night at the table;
- **late**: the fraction is pinned 

The rules you will actually use daily:

- P(not A) = 1 − P(A) 
- P(A and B) = P(A) × P(B) **only when A and B are independent**
- **conditional** probability P(A|B) — of A, *given B happened* —
  = P(A∩B)/P(B)

Counting bigger spaces without listing them *(slide 90)*:
**permutations** count ordered arrangements, **combinations** unordered
ones — `math.perm` and `math.comb`:

#### Order matters or it doesn't — that is the whole difference:

- Permutation — a podium: gold-Ana, silver-Ben is different from gold-Ben, silver-Ana. (A phone PIN too: 1-2-3-4 ≠ 4-3-2-1 — a "combination lock" is really a permutation lock.)
    - Standard notation $n P r$ (or $n$ permute $r$) $$P(n,r) = \dfrac{n!}{(n-r)!}$$
- Combination — a team or pizza toppings: {Ana, Ben} is the same pair however you say it.
    - Standard notation $n$ pick $r$ or $\binom{n}{r}$ ($n$ choose $r$) $$C(n,r) = \binom{n}{r} = \dfrac{n!}{r!(n-r)!}$$

In [ ]:
from math import comb, perm
print("podium orders from 12 students   :", perm(12, 3))
print("project trios from 12 students   :", comb(12, 3))

> **Checkpoint.** To the chat: P(at least one six in two dice rolls) - hint, compute the complement first.

## 2. Random variables  *(slides 68, 69)*

A **random variable** attaches a number to each random outcome — the sum
of two dice, the number of no-shows tomorrow, a patient's blood pressure.
The deck's split *(slide 69)*:

- **discrete** — countable values; described by a *probability mass
  function* (a bar per value) -- how many texts land on your phone in an hour. Only
  whole numbers happen: 0, 1, 2… never 2.5 texts.
- **continuous** — any value in a range; described by a *probability
  density function* (a curve; probability is **area** under it, and the
  probability of any exact single value is zero) -- how long you wait for a bus. Any value in a range:
  3.2 min, 3.24 min, 3.241 min — infinite precision is possible even if
  your watch can't show it.

The distribution zoo *(slides 74, 75)* is a set of named, reusable
shapes. The four you will actually meet, each with its one-line job:
| Distribution | Everyday example | One trial or many? |
|---|---|---|
| **Bernoulli** | one coin flip — heads or tails | exactly 1 trial |
| **Binomial** | number of heads in 10 coin flips | fixed n trials |
| **Poisson** | customers walking into a shop per hour | a rate, no fixed n |
| **Normal** | adult heights in a population | sum of many small effects |

Notice Bernoulli and Binomial are the same coin, different question —
Bernoulli asks about *one* flip, Binomial asks *how many heads* over
several flips. In fact Bernoulli **is** Binomial with n=1.

`scipy.stats` speaks all of them with one grammar — four verbs, each
answering a different question:

| Verb | Question it answers | Works on |
|---|---|---|
| `.rvs(size=…)` | "give me fake data drawn from this shape" | both |
| `.pmf(k)` | "what is P(X **equals exactly** k)?" | discrete only |
| `.pdf(x)` | "how *dense* is probability near x?" (a height, **not** a probability) | continuous only |
| `.cdf(x)` | "what is P(X **≤** x)?" — everything accumulated from the left | both |

Two combinations of `.cdf` cover most real questions:

- **P(X > x)** = `1 - .cdf(x)` — the right tail
- **P(a < X ≤ b)** = `.cdf(b) - .cdf(a)` — a slice

Watch the grammar answer three different questions about one
distribution — a call centre averaging 2.5 calls per minute:

In [ ]:
# Same distribution, three different questions.
# Poisson(mu=2.5) = "calls arriving at 2.5 per minute on average".

p_exactly_3 = stats.poisson.pmf(3, mu=2.5)       # P(X == 3), one bar
p_up_to_3   = stats.poisson.cdf(3, mu=2.5)       # P(X <= 3) = P(0)+P(1)+P(2)+P(3)
p_more_3    = 1 - stats.poisson.cdf(3, mu=2.5)   # P(X > 3), the right tail

print("P(exactly 3 calls) :", round(p_exactly_3, 3))
print("P(3 or fewer)      :", round(p_up_to_3, 3))
print("P(more than 3)     :", round(p_more_3, 3))

Three numbers, three verbs. `pmf` answers *exactly* (0.214 — one bar's
height). `cdf` answers *up to and including* (0.758 — four bars added
together). `1 - cdf` answers *more than* (0.242 — everything else). The
last two add to 1 because "3 or fewer" and "more than 3" split every
possible outcome between them, with nothing shared and nothing missed.

Now the same verbs draw whole distributions. One warning before you
look: the first two panels are **discrete** (bars, `.pmf`), the third is
**continuous** (a curve, `.pdf`) — and the rules for reading them differ.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3))

# --- Bernoulli: ONE coin flip. Only two outcomes exist, so only two bars.
axes[0].bar([0, 1], stats.bernoulli.pmf([0, 1], p=0.5), tick_label=["tails", "heads"])
axes[0].set_title("Bernoulli(0.5): one coin flip")

# --- Binomial: the SAME coin, flipped 10 times, counting heads.
k10 = np.arange(0, 11)
axes[1].bar(k10, stats.binom.pmf(k10, n=10, p=0.5))
axes[1].set_title("Binomial(10, 0.5): heads in 10 flips")

# --- Poisson: a rate, not a trial count. Average 3 customers/hour;
#     the actual count each hour bounces around that average.
k8 = np.arange(0, 8)
axes[2].bar(k8, stats.poisson.pmf(k8, mu=3))
axes[2].set_title("Poisson(3): customers per hour")

# --- Normal: continuous, so a curve. Adult height, mean 170cm, sd 7cm.
xs = np.linspace(145, 195, 200)
axes[3].plot(xs, stats.norm.pdf(xs, loc=170, scale=7))
axes[3].set_title("Normal(170, 7): adult height (cm)")

for ax in axes: ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

Read left to right and watch the shape change:

- **Bernoulli** — two bars, nothing more. Half-and-half here because a
  fair coin is 50/50; an unfair coin would just tilt the two bars.
- **Binomial** — the same coin now flipped 10 times, so possible
  outcomes are 0 through 10 heads. Peaks in the middle at 5 heads
  (P ≈ 0.25) because "half heads" is the most likely single count —
  but far from guaranteed.
- **Poisson** — no coin at all, just a rate. "3 customers per hour on
  average" does **not** mean every hour has exactly 3; 2 and 3 are
  about equally likely (≈ 0.22 each), and 0 customers happens 5% of the
  time.
- **Normal** — a curve, not bars, because height is continuous. The
  peak sits at the mean (170cm)


## 3. The central limit theorem: why the bell is everywhere  *(slide 82)*

The theorem in one sentence, before any machinery:

> **Add up (or average) many small independent random effects, and the
> result comes out bell-shaped — no matter what shape each effect has
> on its own.**



> **Predict first.** length_of_service is heavily right-skewed - Monday's histogram. If we take 2,000 different samples of 40 employees and histogram the 2,000 sample MEANS, what shape appears - same skew, or something else?
>
> Put your answer in the chat before we run it.

In [ ]:
service = hr["length_of_service"].to_numpy()

sample_means = [rng.choice(service, 40).mean() for _ in range(2000000)]

# One sample: grab 40 employees at random, average their service.
# sample_means = []
# for _ in range(2000000):
#     sample = rng.choice(service, 40)    # 40 random employees
#     sample_means.append(sample.mean())  # keep just their mean

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(service, bins=40, edgecolor="white")
axes[0].set_title("The population: skewed")
axes[1].hist(sample_means, bins=40, edgecolor="white", color="tab:green")
axes[1].set_title("2,000 means of samples of 40: a bell")
for ax in axes: ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

A bell, out of nowhere. That is the **central limit theorem**: average
enough independent draws and you get a normal curve — no matter what
the original data looked like. This is why the normal curve is
everywhere in statistics: not because nature is normal, but because
**means are**.

How wide is that bell? The formula needs one distinction first — the
difference between the whole jar and the handful:

### Sample vs. population, before the SEM formula

**Population** = every candy in the whole jar. **Sample** = the one
handful you grabbed to peek at, because counting the whole jar would
take forever. You want to know things about the *whole jar* (its true
average, how spread out it is) but you only ever hold a *handful* —
so you use the handful to make your best guess.

That distinction gives standard deviation two different formulas:

- **Population** (you have *everything*, no guessing):
$$\sigma = \sqrt{\frac{\sum (x_i - \mu)^2}{N}}$$
  $\mu$ is the *true* jar average — known, because you counted every
  candy.

- **Sample** (you only have a handful, must guess):
$$s = \sqrt{\frac{\sum (x_i - \bar{x})^2}{n - 1}}$$
  $\bar{x}$ is your handful's own average — a *guess* at $\mu$, not the
  real thing.

**Why $n-1$, not just $n$?** Your handful's average $\bar{x}$ was
picked to sit as close to your handful's own numbers as possible — the
best possible fit to itself. That makes the handful look *slightly*
less spread out than the real jar actually is. Dividing by $n-1$
instead of $n$ nudges the answer back up to correct for that — a fix
called **Bessel's correction**. This is exactly why the code below uses
`.std(ddof=1)`: `service` is a *sample*, not the whole jar, so it needs
the correction.

That bell of means has a predictable width, the **standard error of
the mean** *(slide 82)*:

$$SEM = \frac{s}{\sqrt{n}}$$

$s$ is the sample standard deviation from the primer above; $n$ is the
sample size. Only one thing to remember from this formula: the
$\sqrt{n}$. 

Keep this one picture in mind, because everything from here builds on
it: **a bell of means, centred on the truth, with width SEM.**

- A **confidence interval** (next) asks: *given my one sample mean,
  where on that bell could the truth be?*
- A **p-value** (section 5) asks the reverse: *assuming a truth, is my
  mean too far out on that bell to be chance?*

In [ ]:
print("predicted SEM  s/sqrt(n):", round(service.std(ddof=1) / np.sqrt(40), 3))
print("observed spread of means:", round(np.std(sample_means, ddof=1), 3))

## 4. Confidence intervals  *(slides 78, 79)*


**mean ± 2 × SEM** is an interval that captures the true population mean
for roughly 95% of samples — a **95% confidence interval**.



In [ ]:
sample = rng.choice(service, 40)
m, sem = sample.mean(), sample.std(ddof=1) / np.sqrt(40)

print("one sample of 40: mean = {:.2f}, SEM = {:.2f}".format(m, sem))
print("95% CI: [{:.2f}, {:.2f}]".format(m - 2 * sem, m + 2 * sem))
print("true population mean:   {:.2f}".format(service.mean()))

## 5. Hypothesis testing: the p-value, built from scratch  *(slides 77, 80, 81, 84)*


Start with a coin, not a formula. A friend flips a coin ten times and
gets **ten heads in a row**. You get suspicious — why? Because a *fair*
coin does that only $(1/2)^{10} = 1/1024 \approx 0.001$ of the time.
- Flip 1: Chance of Heads = \(1/2\)
- Flip 2: Chance of Heads = \(1/2\)
- Flip 3: Chance of Heads = \(1/2\)
- ... and so on, up to 10 times.
- To find the probability of all 10 happening together, you multiply the chances:
- $\frac{1}{2}\times \frac{1}{2}\times \frac{1}{2}\times \dots \times \frac{1}{2}=\left(\frac{1}{2}\right)^{10}$

You just computed a p-value:

> **The p-value = the probability that pure chance produces something
> at least this extreme, assuming nothing real is going on.**

Small p-value → "chance almost never does this" → suspect the coin.
That is the entire idea. Everything below dresses it in formal clothes.

The formal version runs like a **courtroom** analogy -- the burden of proof the right way round:

| Courtroom | Hypothesis test |
|---|---|
| Defendant presumed innocent | **H₀**: no real effect — it's all chance |
| Evidence presented | The data |
| "How surprising is this evidence, *if* innocent?" | The **p-value** |
| Guilty verdict | Reject H₀ — evidence too unlikely under chance |
| "Not proven" — never "proven innocent" | Large p **fails to reject** H₀; it does not prove H₀ |



Live case from our data: does **age at termination** differ from age in
general? 

1. **Null hypothesis H₀** — assume *no real difference*: terminated
   employees are just a random draw from everyone.
2. Compute how big a difference chance alone produces under H₀.
3. If the observed difference would be **very unlikely** under H₀,
   reject H₀. "Very unlikely" is quantified by the **p-value**
   *(slide 77)*: the probability, under H₀, of a difference at least
   this extreme.



## 7. Bayes' theorem: updating beliefs with evidence  *(slide 91)*

You already run Bayes' theorem daily without noticing. A fire alarm
goes off in the shopping centre — do you run? Almost nobody does,
because alarms are nearly always drills or burnt toast: real fires are
*rare*, and your brain quietly weighs that rarity against the evidence.
Bayes' theorem is that instinct written down — with the exact exchange
rate between old belief and new evidence.

The finale also flips tonight's question. Everything so far asked
P(data | no effect). Bayes asks what you actually want:
**P(hypothesis | evidence)**:

$$P(H \mid E) = \frac{P(E \mid H) \times P(H)}{P(E)}$$

Four parts, each with a name and a plain-words job:

| Symbol | Name | Plain words |
|---|---|---|
| $P(H)$ | **prior** | your belief *before* the evidence |
| $P(E \mid H)$ | **likelihood** | if the hypothesis were true, how likely is this evidence? |
| $P(E)$ | **evidence** | how often this evidence shows up at all, from any cause |
| $P(H \mid E)$ | **posterior** | your belief *after* the evidence — the answer |

One warning before the worked case: $P(H \mid E)$ and $P(E \mid H)$
are **not the same number**, and swapping them is the single most
common probability error in real decisions. You are about to see how
different they can be.


---

## Your turn

Official lab: the hypothesis-testing lab in Classroom. Warm-ups:

In [ ]:
# 1. Dice, one line each with the omega set from section 1:
#    P(sum >= 10), and P(sum >= 10 | first die is a 6).

In [ ]:
# 2. A call centre averages 4 calls per minute. Using stats.poisson,
#    what is P(8 or more calls in a minute)?  (Hint: 1 - cdf(7).)

In [ ]:
k = np.arange(0,20)
pmf = stats.poisson.pmf(k, mu = 4)
plt.bar(k, pmf, color = 'red')
print(1-stats.poisson.cdf(4, mu =4))

In [ ]:
# 3. Build a 95% confidence interval for mean age from one random
#    sample of 50 employees (use random_state or rng so it reproduces).
#    Does it contain hr["age"].mean()?

In [ ]:
# 4. t-test: does length_of_service differ between the two genders in
#    this dataset? Save the p-value as `p_gender`, then write one honest
#    sentence: significant or not at alpha = 0.05, and is the effect big?

### Stretch

1. Monday's stretch made you eyeball ACTIVE vs TERMINATED age
   histograms. Re-do the section-5 shuffle test but for the *median*
   instead of the mean - no formula exists, and your simulation does not
   care. That indifference is the superpower of the method.
2. The screening example, but the patient is from a high-risk group
   where prevalence is 1 in 50. Recompute P(sick | positive). What
   changed, and what does that say about screening policy?
3. stats.binom: if true attrition risk is 3% per employee per year,
   what is the probability a 60-person department sees 5 or more
   departures in a year purely by chance?

### If you want the pictures again, slower

Same see-it-first spirit as the whole of Part 2, all free:

- **Seeing Theory** (Brown University) —
  [seeing-theory.brown.edu](https://seeing-theory.brown.edu): its
  interactive Galton board, confidence-interval and Bayesian-inference
  chapters are tonight's sections 3, 4 and 7, with the dials in your
  hands.
- **StatQuest with Josh Starmer** —
  [youtube.com/@statquest](https://www.youtube.com/@statquest): p-values,
  hypothesis testing and the CLT, each in one short rigorous video.
- **3Blue1Brown** —
  [3blue1brown.com/topics/probability](https://www.3blue1brown.com/topics/probability):
  the Bayes' theorem and binomial-distribution animations are the best
  fifteen minutes you can spend before Module 5 meets Naive Bayes.

---

*Data Science & AI — Session 9. Covers Module 1 Part 2 slides 68–91.
Module 2 (EDA) begins Saturday.*